# 102 — TCGA RNA-seq Cohort Freeze

## Objective

This notebook freezes the exact TCGA primary tumor RNA-seq cohort used as the tumor discovery input for the project.

The cohort was constructed manually through the GDC Data Portal using file-level filters designed to retrieve open-access TCGA RNA-seq gene expression quantification files generated with the STAR Counts workflow.

Because GDC queries are dynamic and may change over time as metadata are updated, this notebook does not rely only on a textual description of the filters. Instead, it generates version-controlled cohort definition files that preserve the exact file-level cohort obtained at download time.

## GDC Cohort Definition

The cohort was obtained from the GDC Data Portal using the following filters:

* Program: TCGA
* Data Category: Transcriptome Profiling
* Data Type: Gene Expression Quantification
* Experimental Strategy: RNA-Seq
* Workflow Type: STAR - Counts
* Access: Open
* Tumor Descriptor: Primary

The resulting file-level cohort contained:

* 10,308 RNA-seq STAR Counts files
* 10,328 associated TCGA cases reported by the GDC portal
* 43.67 GB total reported file size

The downloaded GDC files used as raw provenance inputs are:

* `gdc_manifest_tcga_primary_tumor_rnaseq_star_counts.txt`
* `gdc_sample_sheet_tcga_primary_tumor_rnaseq_star_counts.tsv`
* `gdc_metadata_tcga_primary_tumor_rnaseq_star_counts.json`

## Outputs

This notebook generates lightweight, version-controlled cohort freeze files under:

`config/manifests/`

The intended outputs are:

* `gdc_manifest_tcga_primary_tumor_rnaseq_star_counts.txt`
* `gdc_sample_sheet_tcga_primary_tumor_rnaseq_star_counts.tsv`
* `gdc_file_index_tcga_primary_tumor_rnaseq_star_counts.tsv`
* `gdc_cohort_freeze_tcga_primary_tumor_rnaseq_star_counts.json`

These files define the exact TCGA RNA-seq cohort used by the project and allow downstream users to reproduce the same file-level download with the GDC Data Transfer Tool, provided that the referenced GDC file UUIDs remain available.

## Scientific Scope

This cohort is used for tumor-level discovery of recurrent transcriptomic programs. It is not used to infer clinical resistance, causal mechanisms, or therapeutic response. Downstream analyses should treat any associations derived from this cohort as computational and hypothesis-generating.

---


In [1]:
# =============================================================================
# 102 — TCGA RNA-seq Cohort Freeze
# Imports and path configuration
# =============================================================================

import json
import shutil
import sys
from pathlib import Path

import pandas as pd

from src.utils.paths import Paths
from src.utils.checksums import calculate_sha256

In [2]:
# =============================================================================
# Project utilities
# =============================================================================

# Allow imports from src/ when running this notebook from the notebooks directory.
PROJECT_ROOT = Path.cwd().resolve()

while PROJECT_ROOT.name != "pancancer-epigenetics":
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not locate project root directory.")
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))

In [3]:
# =============================================================================
# Raw GDC provenance inputs
# =============================================================================

RAW_TCGA_DIR = Paths.tcga

RAW_MANIFEST_PATH = RAW_TCGA_DIR / "gdc_manifest_tcga_primary_tumor_rnaseq_star_counts.txt"
RAW_SAMPLE_SHEET_PATH = RAW_TCGA_DIR / "gdc_sample_sheet_tcga_primary_tumor_rnaseq_star_counts.tsv"
RAW_METADATA_PATH = RAW_TCGA_DIR / "gdc_metadata_tcga_primary_tumor_rnaseq_star_counts.json"

In [4]:
# =============================================================================
# Version-controlled cohort freeze outputs
# =============================================================================

CONFIG_MANIFEST_DIR = Paths.config / "manifests"
CONFIG_MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

OUT_MANIFEST_PATH = CONFIG_MANIFEST_DIR / "gdc_manifest_tcga_primary_tumor_rnaseq_star_counts.txt"
OUT_SAMPLE_SHEET_PATH = CONFIG_MANIFEST_DIR / "gdc_sample_sheet_tcga_primary_tumor_rnaseq_star_counts.tsv"
OUT_FILE_INDEX_PATH = CONFIG_MANIFEST_DIR / "gdc_file_index_tcga_primary_tumor_rnaseq_star_counts.tsv"
OUT_COHORT_FREEZE_PATH = CONFIG_MANIFEST_DIR / "gdc_cohort_freeze_tcga_primary_tumor_rnaseq_star_counts.json"

In [5]:
# =============================================================================
# Validate raw input availability
# =============================================================================

raw_input_paths = {
    "manifest": RAW_MANIFEST_PATH,
    "sample_sheet": RAW_SAMPLE_SHEET_PATH,
    "metadata_json": RAW_METADATA_PATH,
}

missing_files = [path for path in raw_input_paths.values() if not path.exists()]

if missing_files:
    raise FileNotFoundError(
        "Missing required raw TCGA provenance files:\n"
        + "\n".join(str(path) for path in missing_files)
    )

for label, path in raw_input_paths.items():
    print(f"{label}: {path.stat().st_size / 1024**2:.2f} MB")

manifest: 1.60 MB
sample_sheet: 2.36 MB
metadata_json: 25.82 MB


In [6]:
# =============================================================================
# Load GDC manifest and sample sheet
# =============================================================================

manifest_df = pd.read_csv(RAW_MANIFEST_PATH, sep="\t")
sample_sheet_df = pd.read_csv(RAW_SAMPLE_SHEET_PATH, sep="\t")

print("Manifest shape:", manifest_df.shape)
print("Sample sheet shape:", sample_sheet_df.shape)

display(manifest_df.head())
display(sample_sheet_df.head())

Manifest shape: (10308, 5)
Sample sheet shape: (10308, 11)


,id,filename,md5,size,state
0,744a6d3d-b666-49aa-8d26-47f34e3d1eb5,94027f46-390c-4dda-ab89-cb1ac0a291cd.rna_seq.a...,4a91710ffccd29966a3374ffaf14abec,4231952,released
1,4ecc1f1a-8ff4-4552-a5e8-7a9652b6d1d5,df45fb41-4511-4dab-b865-fcdfcda0400a.rna_seq.a...,af51e54b9dc2f6aecd39ab26d2b79824,4238832,released
2,1ace2a0c-773d-45b5-8fd6-968c88731bbb,c33ecc28-d5ba-4416-b93d-445b7883b6e8.rna_seq.a...,ff767304460d3385c372d7e0818cb45a,4239626,released
3,7db46624-6d07-4f49-b0ff-18bb41375de9,b1f7af2a-a39b-44c7-a077-e1df6b18268a.rna_seq.a...,82efe74a8659e27431b3a6c2970b9dcb,4236752,released
4,25e923e1-24ce-44a9-934c-20df73d33686,acfb6bc0-2972-48f8-9926-5a5ef46e8d86.rna_seq.a...,7d9634f48654e3a41c5d1e192a6180e6,4229913,released


,File ID,File Name,Data Category,Data Type,Project ID,Case ID,Sample ID,Tissue Type,Tumor Descriptor,Specimen Type,Preservation Method
0,744a6d3d-b666-49aa-8d26-47f34e3d1eb5,94027f46-390c-4dda-ab89-cb1ac0a291cd.rna_seq.a...,Transcriptome Profiling,Gene Expression Quantification,TCGA-BRCA,TCGA-BH-A18H,TCGA-BH-A18H-01A,Tumor,Primary,Solid Tissue,OCT
1,4ecc1f1a-8ff4-4552-a5e8-7a9652b6d1d5,df45fb41-4511-4dab-b865-fcdfcda0400a.rna_seq.a...,Transcriptome Profiling,Gene Expression Quantification,TCGA-BRCA,TCGA-E2-A14P,TCGA-E2-A14P-01A,Tumor,Primary,Solid Tissue,Unknown
2,1ace2a0c-773d-45b5-8fd6-968c88731bbb,c33ecc28-d5ba-4416-b93d-445b7883b6e8.rna_seq.a...,Transcriptome Profiling,Gene Expression Quantification,TCGA-BRCA,TCGA-AN-A04A,TCGA-AN-A04A-01A,Tumor,Primary,Solid Tissue,OCT
3,7db46624-6d07-4f49-b0ff-18bb41375de9,b1f7af2a-a39b-44c7-a077-e1df6b18268a.rna_seq.a...,Transcriptome Profiling,Gene Expression Quantification,TCGA-KIRP,TCGA-MH-A855,TCGA-MH-A855-01A,Tumor,Primary,Solid Tissue,OCT
4,25e923e1-24ce-44a9-934c-20df73d33686,acfb6bc0-2972-48f8-9926-5a5ef46e8d86.rna_seq.a...,Transcriptome Profiling,Gene Expression Quantification,TCGA-KIRP,TCGA-P4-A5E6,TCGA-P4-A5E6-01A,Tumor,Primary,Solid Tissue,OCT


In [7]:
# =============================================================================
# Validate manifest and sample sheet structure
# =============================================================================

expected_manifest_columns = {"id", "filename", "md5", "size", "state"}
expected_sample_sheet_columns = {
    "File ID",
    "File Name",
    "Data Category",
    "Data Type",
    "Project ID",
    "Case ID",
    "Sample ID",
    "Tissue Type",
    "Tumor Descriptor",
    "Specimen Type",
    "Preservation Method",
}

missing_manifest_columns = expected_manifest_columns - set(manifest_df.columns)
missing_sample_sheet_columns = expected_sample_sheet_columns - set(sample_sheet_df.columns)

if missing_manifest_columns:
    raise ValueError(f"Missing manifest columns: {sorted(missing_manifest_columns)}")

if missing_sample_sheet_columns:
    raise ValueError(
        f"Missing sample sheet columns: {sorted(missing_sample_sheet_columns)}"
    )

print("Manifest columns validated.")
print("Sample sheet columns validated.")

Manifest columns validated.
Sample sheet columns validated.


In [8]:
# =============================================================================
# Validate manifest and sample sheet consistency
# =============================================================================

manifest_file_ids = set(manifest_df["id"])
sample_sheet_file_ids = set(sample_sheet_df["File ID"])

only_in_manifest = manifest_file_ids - sample_sheet_file_ids
only_in_sample_sheet = sample_sheet_file_ids - manifest_file_ids

print(f"Files in manifest: {len(manifest_file_ids):,}")
print(f"Files in sample sheet: {len(sample_sheet_file_ids):,}")
print(f"File IDs only in manifest: {len(only_in_manifest):,}")
print(f"File IDs only in sample sheet: {len(only_in_sample_sheet):,}")

if only_in_manifest or only_in_sample_sheet:
    raise ValueError("Manifest and sample sheet do not contain the same File IDs.")

file_name_check_df = manifest_df[["id", "filename"]].merge(
    sample_sheet_df[["File ID", "File Name"]],
    left_on="id",
    right_on="File ID",
    how="inner",
    validate="one_to_one",
)

n_filename_mismatches = (
    file_name_check_df["filename"] != file_name_check_df["File Name"]
).sum()

print(f"Filename mismatches: {n_filename_mismatches:,}")

if n_filename_mismatches:
    raise ValueError("Manifest and sample sheet contain mismatched filenames.")

print("Manifest and sample sheet define the same file-level cohort.")

Files in manifest: 10,308
Files in sample sheet: 10,308
File IDs only in manifest: 0
File IDs only in sample sheet: 0
Filename mismatches: 0
Manifest and sample sheet define the same file-level cohort.


In [9]:
# =============================================================================
# Validate frozen cohort definition
# =============================================================================

expected_filename_suffix = ".rna_seq.augmented_star_gene_counts.tsv"

checks = {
    "all_star_counts": manifest_df["filename"].str.endswith(
        expected_filename_suffix, na=False
    ).all(),
    "all_released": (manifest_df["state"] == "released").all(),
    "all_transcriptome_profiling": (
        sample_sheet_df["Data Category"] == "Transcriptome Profiling"
    ).all(),
    "all_gene_expression_quantification": (
        sample_sheet_df["Data Type"] == "Gene Expression Quantification"
    ).all(),
    "all_tumor": (sample_sheet_df["Tissue Type"] == "Tumor").all(),
    "all_primary": (sample_sheet_df["Tumor Descriptor"] == "Primary").all(),
}

for check_name, passed in checks.items():
    print(f"{check_name}: {passed}")

failed_checks = [name for name, passed in checks.items() if not passed]

if failed_checks:
    raise ValueError(
        "Frozen TCGA cohort failed expected definition checks: "
        + ", ".join(failed_checks)
    )

print("Frozen cohort definition validated.")

all_star_counts: True
all_released: True
all_transcriptome_profiling: True
all_gene_expression_quantification: True
all_tumor: True
all_primary: True
Frozen cohort definition validated.


In [10]:
# =============================================================================
# Build frozen TCGA RNA-seq file index
# =============================================================================

manifest_index_df = manifest_df.rename(
    columns={
        "id": "file_id",
        "filename": "file_name",
        "md5": "md5",
        "size": "file_size_bytes",
        "state": "state",
    }
)

sample_sheet_index_df = sample_sheet_df.rename(
    columns={
        "File ID": "file_id",
        "File Name": "file_name_sample_sheet",
        "Data Category": "data_category",
        "Data Type": "data_type",
        "Project ID": "project_id",
        "Case ID": "case_id",
        "Sample ID": "sample_id",
        "Tissue Type": "tissue_type",
        "Tumor Descriptor": "tumor_descriptor",
        "Specimen Type": "specimen_type",
        "Preservation Method": "preservation_method",
    }
)

file_index_df = manifest_index_df.merge(
    sample_sheet_index_df,
    on="file_id",
    how="inner",
    validate="one_to_one",
)

file_name_mismatches = (
    file_index_df["file_name"] != file_index_df["file_name_sample_sheet"]
).sum()

if file_name_mismatches:
    raise ValueError("Manifest and sample sheet filenames do not match.")

file_index_df = file_index_df.drop(columns=["file_name_sample_sheet"])

file_index_df = file_index_df[
    [
        "file_id",
        "file_name",
        "md5",
        "file_size_bytes",
        "state",
        "project_id",
        "case_id",
        "sample_id",
        "data_category",
        "data_type",
        "tissue_type",
        "tumor_descriptor",
        "specimen_type",
        "preservation_method",
    ]
].sort_values(["project_id", "case_id", "sample_id", "file_id"])

print("File index shape:", file_index_df.shape)
print("Unique projects:", file_index_df["project_id"].nunique())
print("Unique cases:", file_index_df["case_id"].nunique())
print("Unique samples:", file_index_df["sample_id"].nunique())

display(file_index_df.head())

File index shape: (10308, 14)
Unique projects: 33
Unique cases: 10122
Unique samples: 10174


,file_id,file_name,md5,file_size_bytes,state,project_id,case_id,sample_id,data_category,data_type,tissue_type,tumor_descriptor,specimen_type,preservation_method
2133,fe16b2d3-17b0-4e24-ab31-62d2e951b3a2,6bacf042-830f-47dd-bac3-5696aebf2574.rna_seq.a...,967a3efd89a8d709df7170d3be032580,4225986,released,TCGA-ACC,TCGA-OR-A5J1,TCGA-OR-A5J1-01A,Transcriptome Profiling,Gene Expression Quantification,Tumor,Primary,Unknown,OCT
2142,f11aa62e-5d67-4e49-8f64-5ed663a3b424,06cc0bc6-1b36-4318-99ee-dfc83cc134c0.rna_seq.a...,9f41ec17c77ac638baaf7d7e065f03df,4233682,released,TCGA-ACC,TCGA-OR-A5J2,TCGA-OR-A5J2-01A,Transcriptome Profiling,Gene Expression Quantification,Tumor,Primary,Unknown,OCT
2199,3137ba2b-0d6b-4fcc-8512-894fd6a43ff6,171e4e82-93ed-4be7-8b61-7c1df82bde0f.rna_seq.a...,d1afd241c711ac71989ea60c1874a536,4222749,released,TCGA-ACC,TCGA-OR-A5J3,TCGA-OR-A5J3-01A,Transcriptome Profiling,Gene Expression Quantification,Tumor,Primary,Unknown,OCT
2181,b8abd8a3-1525-4bf5-b083-eddefa3ca19e,32d1a985-ad2c-4046-873d-02416f5a68f3.rna_seq.a...,2b2d56f116f7d3d78f008ee80b73d772,4219218,released,TCGA-ACC,TCGA-OR-A5J5,TCGA-OR-A5J5-01A,Transcriptome Profiling,Gene Expression Quantification,Tumor,Primary,Unknown,OCT
2193,224fa76c-cf8d-48d8-8723-15907a04f2d2,a0169877-dd20-4c7d-826c-bb424199a036.rna_seq.a...,ba6a678a758ed96e14768cb2c50bb3a8,4219826,released,TCGA-ACC,TCGA-OR-A5J6,TCGA-OR-A5J6-01A,Transcriptome Profiling,Gene Expression Quantification,Tumor,Primary,Unknown,OCT


In [11]:
# =============================================================================
# Define cohort-freeze metadata
# =============================================================================
cohort_freeze = {
    "cohort_name": "tcga_primary_tumor_rnaseq_star_counts",
    "source_database": "Genomic Data Commons (GDC)",
    "provider": "National Cancer Institute",
    "retrieval_method": "GDC Data Portal query-derived cohort export",
    "download_page_url": "https://portal.gdc.cancer.gov/repository",
    "download_date": "2026-06-12",
    "filters": {
        "program": "TCGA",
        "data_category": "Transcriptome Profiling",
        "data_type": "Gene Expression Quantification",
        "experimental_strategy": "RNA-Seq",
        "workflow_type": "STAR - Counts",
        "access": "Open",
        "tissue_type": "Tumor",
        "tumor_descriptor": "Primary",
    },
    "raw_inputs": {
        "manifest": {
            "path": str(RAW_MANIFEST_PATH.relative_to(Paths.root)),
            "sha256": calculate_sha256(RAW_MANIFEST_PATH),
        },
        "sample_sheet": {
            "path": str(RAW_SAMPLE_SHEET_PATH.relative_to(Paths.root)),
            "sha256": calculate_sha256(RAW_SAMPLE_SHEET_PATH),
        },
        "metadata_json": {
            "path": str(RAW_METADATA_PATH.relative_to(Paths.root)),
            "sha256": calculate_sha256(RAW_METADATA_PATH),
        },
    },
    "cohort_counts": {
        "n_files": int(file_index_df["file_id"].nunique()),
        "n_projects": int(file_index_df["project_id"].nunique()),
        "n_cases": int(file_index_df["case_id"].nunique()),
        "n_samples": int(file_index_df["sample_id"].nunique()),
        "total_file_size_bytes": int(file_index_df["file_size_bytes"].sum()),
        "total_file_size_gib": round(
            file_index_df["file_size_bytes"].sum() / 1024**3, 3
        ),
    },
    "versioned_outputs": {
        "manifest": str(OUT_MANIFEST_PATH.relative_to(Paths.root)),
        "sample_sheet": str(OUT_SAMPLE_SHEET_PATH.relative_to(Paths.root)),
        "file_index": str(OUT_FILE_INDEX_PATH.relative_to(Paths.root)),
        "cohort_freeze": str(OUT_COHORT_FREEZE_PATH.relative_to(Paths.root)),
    },
    "reproducibility_note": (
        "This TCGA cohort was generated from a GDC Data Portal query. "
        "Because portal queries are dynamic, the version-controlled manifest "
        "and file index define the exact frozen file-level cohort used by this repository."
    ),
    "scientific_scope": (
        "Tumor-level discovery input for recurrent transcriptomic program analysis. "
        "This cohort freeze does not imply clinical resistance prediction, causal inference, "
        "or therapeutic validation."
    ),
}

print(json.dumps(cohort_freeze, indent=2))

{
  "cohort_name": "tcga_primary_tumor_rnaseq_star_counts",
  "source_database": "Genomic Data Commons (GDC)",
  "provider": "National Cancer Institute",
  "retrieval_method": "GDC Data Portal query-derived cohort export",
  "download_page_url": "https://portal.gdc.cancer.gov/repository",
  "download_date": "2026-06-12",
  "filters": {
    "program": "TCGA",
    "data_category": "Transcriptome Profiling",
    "data_type": "Gene Expression Quantification",
    "experimental_strategy": "RNA-Seq",
    "workflow_type": "STAR - Counts",
    "access": "Open",
    "tissue_type": "Tumor",
    "tumor_descriptor": "Primary"
  },
  "raw_inputs": {
    "manifest": {
      "path": "data\\raw\\tcga\\gdc_manifest_tcga_primary_tumor_rnaseq_star_counts.txt",
      "sha256": "3fcbb6353753d47a851826005500ca6de7f58a3734479075e390bd33e712cc3d"
    },
    "sample_sheet": {
      "path": "data\\raw\\tcga\\gdc_sample_sheet_tcga_primary_tumor_rnaseq_star_counts.tsv",
      "sha256": "bcb922f064d2aa4c60c4f10f1a